In [ ]:
import h5py
import numpy as np
import json
import matplotlib.pyplot as plt

from numba.typed import Dict
from numba import types
import numba as nb

In [ ]:
f = h5py.File('/home/yousen/Public/ndlar_shared/data/diagnostics/ndlar_flow/packet-2024_06_28_08_51_52_CDT.FLOW.hdf5')

In [ ]:
def dereference(A_data, B_data, ref_AtoB):
    return A_data[:][ref_AtoB[:,0]], B_data[:][ref_AtoB[:,1]]

charge, packet = dereference(f['charge/raw_hits/data'], f['charge/packets/data'], f['charge/raw_hits/ref/charge/packets/ref'])

In [ ]:
len(charge)

In [ ]:
def hit_unique_str(packets_arr):
    # io group to tile id
    tile_id = 1+(packets_arr['io_group']-1)*8+(packets_arr['io_channel']-1)//4 # 8 and 4 are hard-wired. 8 io groups and 4 io channels per tile
    hit_uniqueid = (packets_arr['io_group'].astype(int)*1000_000_000
                            + tile_id.astype(int)*100_000
                            + packets_arr['chip_id'].astype(int)*100
                            + packets_arr['channel_id'].astype(int))
    return hit_uniqueid

In [ ]:
hid = hit_unique_str(packet)

In [ ]:
fpath = '/home/yousen/Public/ndlar_shared/data/diagnostics/ndlar_flow/reference-cold-pedestal-2024_06_05_08_28_19_CDTevd_ped.tile_id.decimal.json'
def load_ped(fpath):
    with open(fpath) as fped:
        dped = json.load(fped)
    d = Dict.empty(key_type=types.int64, value_type=types.float64)
    for k, v in dped.items():
        d[int(k)] = v['pedestal_mv']
    return d
dped = load_ped(fpath)

In [ ]:
len(dped)/39200

In [ ]:
@nb.njit
def lookup(arr, d, default=np.float32(-1.0)):
    result = np.empty(arr.shape[0], dtype=np.float32)
    mask = np.zeros(arr.shape[0], dtype=np.bool_)

    for i in range(arr.shape[0]):
        val = d.get(arr[i], default)
        result[i] = val
        if abs(val - default) < 1e-5:  # Safe float comparison
            mask[i] = True

    return result, mask

In [ ]:
ped, ped_mask = lookup(hid, dped)

In [ ]:
len(np.nonzero(ped_mask)[0])

In [ ]:
11990647/len(charge)

In [ ]:
def charge_from_dataword(dw, vref, vcm, ped, adc_counts, gain):
    return (dw / adc_counts * (vref - vcm) + vcm - ped) / gain

In [ ]:
q = charge_from_dataword(charge['ADC'], vref=1568.0, vcm=478.1, ped=ped, adc_counts=256, gain=4.522) # 4.522

In [ ]:
import numpy.lib.recfunctions as rfn
with h5py.File('threshold_input.hdf5', 'w') as fh5:
    charge = charge[~ped_mask]
    packet = packet[~ped_mask]
    q = q[~ped_mask]
    ped = ped[~ped_mask]
    for ig in list(range(1,9)):
        args = np.nonzero(packet['io_group'] == ig)[0]
        hits = charge[args]
        pk = packet[args]
        qs = q[args]
        p = ped[args]
        hits = rfn.append_fields(hits, names=['Q', 'ped'], data=[qs, p], usemask=False)
        hits = rfn.append_fields(hits, names=['io_group', 'io_channel', 'chip_id', 'channel_id'], 
                                 data=[pk['io_group'], pk['io_channel'], pk['chip_id'], pk['channel_id'] ], usemask=False)
        fh5.create_dataset(f'/io_group{ig}/hits', data=hits)